In [1]:
# !pip install tensorflow==2.3.0
!pip install tensorflow==2.12.0
!pip install gym==0.25.2
!pip install keras
!pip install keras-rl2

  Using cached tensorflow-2.12.0-cp310-cp310-win_amd64.whl.metadata (2.5 kB)
Using cached tensorflow-2.12.0-cp310-cp310-win_amd64.whl (1.9 kB)
   ---------------------------------------- 0.0/1.7 MB ? eta -:--:--
   ---------------------------------------- 0.0/1.7 MB ? eta -:--:--
   ---------------------------------------- 0.0/1.7 MB ? eta -:--:--
   ------ --------------------------------- 0.3/1.7 MB ? eta -:--:--
   ------ --------------------------------- 0.3/1.7 MB ? eta -:--:--
   ------ --------------------------------- 0.3/1.7 MB ? eta -:--:--
   ------ --------------------------------- 0.3/1.7 MB ? eta -:--:--
   ------------ --------------------------- 0.5/1.7 MB 212.4 kB/s eta 0:00:06
   ------------ --------------------------- 0.5/1.7 MB 212.4 kB/s eta 0:00:06
   ------------ --------------------------- 0.5/1.7 MB 212.4 kB/s eta 0:00:06
   ------------------------ --------------- 1.0/1.7 MB 357.0 kB/s eta 0:00:02
   ------------------------ --------------- 1.0/1.7 MB 357.0 k

In [2]:
import gym 
import random
import numpy as np
import tensorflow
from keras.models import Sequential
from keras.layers import Dense, Flatten
from keras.optimizers import Adam

C:\Users\RAB\AppData\Roaming\Python\Python310\site-packages\tensorflow\lite\python\util.py:52: DeprecationWarning: jax.xla_computation is deprecated. Please use the AOT APIs.
  from jax import xla_computation as _xla_computation


In [3]:
print(gym.__version__)
print(tensorflow.__version__)
print(tensorflow.keras.__version__)

0.25.2
2.12.0
2.12.0


In [4]:
env = gym.make('CartPole-v1',render_mode="human")
 
actions = env.action_space.n
states = env.observation_space.shape[0]

c:\Users\RAB\AppData\Local\Programs\Python\Python310\lib\site-packages\gym\core.py:317: DeprecationWarning: WARN: Initializing wrapper in old step API which returns one bool instead of two. It is recommended to set `new_step_api=True` to use new step API. This will be the default behaviour in future.
  deprecation(
c:\Users\RAB\AppData\Local\Programs\Python\Python310\lib\site-packages\gym\wrappers\step_api_compatibility.py:39: DeprecationWarning: WARN: Initializing environment in old step API which returns one bool instead of two. It is recommended to set `new_step_api=True` to use new step API. This will be the default behaviour in future.
  deprecation(


In [5]:
env.reset()

array([0.03676718, 0.00568856, 0.03688453, 0.03936956], dtype=float32)

In [6]:
states  ,actions
 

(4, 2)

In [ ]:
action =1
print(action)
new_state, reward, done, info = env.step(action)
env.render()
print(new_state, reward, done, info) 

1
[ 0.03688095  0.20026271  0.03767192 -0.24145156] 1.0 False {}


In [8]:
episodes = 10
for episode in range(1, episodes+1):
    state = env.reset()
    done = False
    score = 0 
    ic=0
    while not done:
        env.render()
        action = random.choice([0,1])
        
        n_state, reward, done, info= env.step(action)
        if (done):
            print(n_state)
         
        score+=reward
         
    print('Episode:{} Score:{}'.format(episode, score))
    

 

[ 0.1216544   0.591004   -0.21776384 -1.0995028 ]
Episode:1 Score:13.0
[ 0.09161068 -0.40230626 -0.21278809  0.01676432]
Episode:2 Score:16.0
[ 0.14505327  1.1412026  -0.23821443 -1.9353738 ]
Episode:3 Score:20.0
[ 0.01128309  0.7537342  -0.22031271 -1.7388732 ]
Episode:4 Score:28.0
[-0.14139573 -1.1754339   0.2140304   1.6470602 ]
Episode:5 Score:36.0
[-0.12324826 -1.1388698   0.23654306  2.0473483 ]
Episode:6 Score:10.0
[ 0.15532638  1.1747681  -0.24446753 -2.174748  ]
Episode:7 Score:20.0
[-0.00630255 -0.4717349   0.2263982   1.6272956 ]
Episode:8 Score:36.0
[-0.16550194 -1.230988    0.22230522  2.0445898 ]
Episode:9 Score:14.0
[ 0.22893421  1.9759817  -0.2142633  -2.9673321 ]
Episode:10 Score:12.0


In [9]:

model =Sequential()
model.add(Flatten(input_shape=(1,states)))
model.add(Dense(24, activation='relu'))
model.add(Dense(24, activation='relu'))
model.add(Dense(actions, activation='linear')) 


In [10]:
model.summary()

Model: "sequential"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 flatten (Flatten)           (None, 4)                 0         
                                                                 
 dense (Dense)               (None, 24)                120       
                                                                 
 dense_1 (Dense)             (None, 24)                600       
                                                                 
 dense_2 (Dense)             (None, 2)                 50        
                                                                 
Total params: 770
Trainable params: 770
Non-trainable params: 0
_________________________________________________________________


In [11]:
from rl.agents import DQNAgent
from rl.policy import BoltzmannQPolicy
from rl.memory import SequentialMemory

In [12]:
def build_agent(model, actions):
    policy = BoltzmannQPolicy()
    memory = SequentialMemory(limit=50000, window_length=1)
    dqn = DQNAgent(model=model, memory=memory, policy=policy, 
                  nb_actions=actions, nb_steps_warmup=10, target_model_update=1e-2)
    return dqn

In [ ]:
dqn = build_agent(model, actions)
dqn.compile(Adam(lr=1e-3), metrics=['mae'])

AttributeError: 'Sequential' object has no attribute '_compile_time_distribution_strategy'

In [15]:
dqn.fit(env , nb_steps=10000, visualize=False, verbose=1)

RuntimeError: Your tried to fit your agent but it hasn't been compiled yet. Please call `compile()` before `fit()`.

In [60]:
scores = dqn.test(env, nb_episodes=10, visualize=False)
print(np.mean(scores.history['episode_reward']))

Testing for 10 episodes ...
Episode 1: reward: 255.000, steps: 255
Episode 2: reward: 258.000, steps: 258
Episode 3: reward: 479.000, steps: 479
Episode 4: reward: 500.000, steps: 500
Episode 5: reward: 290.000, steps: 290
Episode 6: reward: 274.000, steps: 274
Episode 7: reward: 273.000, steps: 273
Episode 8: reward: 500.000, steps: 500
Episode 9: reward: 253.000, steps: 253
Episode 10: reward: 346.000, steps: 346
342.8


In [ ]:
_ = dqn.test(env, nb_episodes=5, visualize=True)

RuntimeError: Your tried to test your agent but it hasn't been compiled yet. Please call `compile()` before `test()`.

: 

In [37]:
env.close()